In [1]:
import numpy as np
import pandas as pd
import os
import copy
import glob
import random
from random import random as rd
import gc



In [2]:
df_train = pd.read_csv("../input/ventilator-pressure-prediction/train.csv")

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else upper_val
    )


def snap_to_pressure_grid(pred: np.ndarray) -> np.ndarray:
    pred = np.asarray(pred, dtype=float)
    idx = np.searchsorted(sorted_pressures, pred, side="left")
    idx = np.clip(idx, 0, total_pressures_len - 1)

    left_idx = np.clip(idx - 1, 0, total_pressures_len - 1)
    right_idx = idx

    left_val = sorted_pressures[left_idx]
    right_val = sorted_pressures[right_idx]

    choose_left = np.abs(pred - left_val) < np.abs(pred - right_val)
    out = np.where(choose_left, left_val, right_val)

    return out.astype(float)


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    l = []
    for i in range(len(input_list)):
        public_lb_score = int(input_list[i].split("/")[-1].split(".")[1].split(" ")[0])
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).ravel()
    output = 0
    l_sum = sum(l)
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = (l[1] / l_sum) + 0.1
        weight2 = 1 - weight1
        output += input_list[0] * weight1 + input_list[1] * weight2
    return output


def g(dp):
    l = []
    for i in glob.iglob(f"{dp}/*"):
        l.append(i)
    file_count = len(l)
    loop_time = 154
    splits = file_count // 2
    l.sort()
    flist = []
    for i in range(splits):
        if i == splits - 1:
            flist.append(l[i * round(len(l) / splits) :])
        else:
            flist.append(
                l[i * round(len(l) / splits) : (i + 1) * round(len(l) / splits)]
            )
    for i in range(len(flist)):
        flist[i] = wc(flist[i])
    pred_list = []
    for i in range(loop_time):
        weight = []
        set_seed(i)
        for j in range(len(flist)):
            weight.append(rd())
        weight_sum = sum(weight)
        for j in range(len(weight)):
            weight[j] /= weight_sum
        weight.sort(reverse=True)
        temp = 0
        for j in range(len(flist)):
            temp += flist[j] * weight[j]
        pred_list.append(temp)
        del temp
        gc.collect()
    output = pd.read_csv(
        "../input/ventilator-pressure-prediction/sample_submission.csv"
    )
    output.pressure = np.median(np.vstack(pred_list), axis=0)
    output["pressure"] = output["pressure"].apply(find_nearest)
    output.to_csv(f"rwb {loop_time} loops.csv", index=False)


def blend(a, b):
    a = pd.read_csv(a)
    b = pd.read_csv(b)
    a.pressure = a.pressure * 0.6 + b.pressure * 0.4
    a["pressure"] = a["pressure"].apply(find_nearest)
    a.to_csv("blend.csv", index=False)
    return a




In [3]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

set_seed(2021)

df_test = pd.read_csv("../input/ventilator-pressure-prediction/test.csv")


def add_features(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    df = df.copy()

    df["_orig_order"] = np.arange(len(df), dtype=np.int64)
    df.sort_values(["breath_id", "time_step"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df["u_in_lag1"] = df.groupby("breath_id")["u_in"].shift(1).fillna(0.0)
    df["u_in_lag2"] = df.groupby("breath_id")["u_in"].shift(2).fillna(0.0)
    df["u_out_lag1"] = (
        df.groupby("breath_id")["u_out"].shift(1).fillna(0).astype(np.int16)
    )

    df["u_in_diff1"] = df["u_in"] - df["u_in_lag1"]
    df["u_in_diff2"] = df["u_in_lag1"] - df["u_in_lag2"]

    df["u_in_cumsum"] = df.groupby("breath_id")["u_in"].cumsum()
    df["u_in_time"] = df["u_in"] * df["time_step"]
    df["u_in_cumsum_time"] = df["u_in_cumsum"] * df["time_step"]

    df["u_in_over_R"] = df["u_in"] / df["R"].astype(float)
    df["u_in_over_C"] = df["u_in"] / df["C"].astype(float)

    df["timestep_idx"] = df.groupby("breath_id").cumcount().astype(np.int16)
    df["is_insp"] = (df["u_out"] == 0).astype(np.int8)

    df["u_in_rollmean_5"] = (
        df.groupby("breath_id")["u_in"]
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )
    df["insp_count"] = df.groupby("breath_id")["is_insp"].cumsum().astype(np.int16)

    u_out_change = df.groupby("breath_id")["u_out"].diff().fillna(0).astype(np.int8)
    opened = (u_out_change == 1).astype(np.int8)
    df["u_out_opened"] = opened
    df["u_out_open_event_id"] = (
        df.groupby("breath_id")["u_out_opened"].cumsum().astype(np.int16)
    )
    df["time_since_u_out_open"] = (
        df.groupby(["breath_id", "u_out_open_event_id"]).cumcount().astype(np.int16)
    )
    df.drop(columns=["u_out_open_event_id"], inplace=True)

    df["time_step_centered"] = (
        df["time_step"] - df.groupby("breath_id")["time_step"].transform("mean")
    ).astype(np.float32)

    df["u_in_is_insp"] = (df["u_in"] * df["is_insp"]).astype(np.float32)

    df["RC"] = (df["R"].astype(str) + "_" + df["C"].astype(str)).astype("category")
    df["R_cat"] = df["R"].astype("category")
    df["C_cat"] = df["C"].astype("category")

    return df


train_feat = add_features(df_train, is_train=True)
test_feat = add_features(df_test, is_train=False)

combined = pd.concat(
    [train_feat[["RC", "R_cat", "C_cat"]], test_feat[["RC", "R_cat", "C_cat"]]], axis=0
)
combined_dummies = pd.get_dummies(
    combined, columns=["RC", "R_cat", "C_cat"], drop_first=False, dtype=np.int8
)

train_dummies = combined_dummies.iloc[: len(train_feat)].reset_index(drop=True)
test_dummies = combined_dummies.iloc[len(train_feat) :].reset_index(drop=True)

train_feat = train_feat.reset_index(drop=True)
test_feat = test_feat.reset_index(drop=True)

train_feat = pd.concat([train_feat, train_dummies], axis=1)
test_feat = pd.concat([test_feat, test_dummies], axis=1)

# CHANGE (score-improvement, metric alignment): train only on inspiratory rows (u_out==0)
# because Kaggle scores only inspiratory phase. Also drop u_out/is_insp-derived predictors from
# the regression inputs (they become constant in training and can add noise at inference).
feature_cols = [
    "time_step",
    "time_step_centered",
    "timestep_idx",
    "u_in",
    "u_in_lag1",
    "u_in_lag2",
    "u_out_lag1",
    "u_in_diff1",
    "u_in_diff2",
    "u_in_cumsum",
    "u_in_time",
    "u_in_cumsum_time",
    "u_in_over_R",
    "u_in_over_C",
    "u_out_opened",
    "time_since_u_out_open",
    "u_in_rollmean_5",
    "insp_count",
]

dummy_cols = [
    c
    for c in train_feat.columns
    if c.startswith("RC_") or c.startswith("R_cat_") or c.startswith("C_cat_")
]
feature_cols = feature_cols + dummy_cols

X_train_all = train_feat[feature_cols].astype(np.float32).values
y_train_all = train_feat["pressure"].astype(np.float32).values
X_test_sorted = test_feat[feature_cols].astype(np.float32).values

insp_mask_train = train_feat["u_out"].values == 0
insp_mask_test_sorted = test_feat["u_out"].values == 0

X_train = X_train_all[insp_mask_train]
y_train = y_train_all[insp_mask_train]
train_groups = train_feat.loc[insp_mask_train, ["R", "C"]].astype(np.int16).values

test_groups = test_feat[["R", "C"]].astype(np.int16).values
unique_groups = np.unique(train_groups, axis=0)

train_insp = train_feat.loc[
    insp_mask_train, ["breath_id", "R", "C", "timestep_idx", "pressure"]
].copy()
last_insp_idx = train_insp.groupby("breath_id")["timestep_idx"].transform("max")
end_insp = train_insp.loc[
    train_insp["timestep_idx"].values == last_insp_idx.values, ["R", "C", "pressure"]
]
end_insp_median_by_group = end_insp.groupby(["R", "C"])["pressure"].median()

global_exp_baseline = (
    float(np.median(end_insp["pressure"].values))
    if len(end_insp)
    else float(sorted_pressures[0])
)
global_exp_baseline = float(find_nearest(global_exp_baseline))

test_pred_sorted = np.full(
    shape=(len(test_feat),), fill_value=global_exp_baseline, dtype=np.float32
)

min_group_train_samples = 2000  # unchanged guardrail

for Rv, Cv in unique_groups:
    tr_g = (train_groups[:, 0] == Rv) & (train_groups[:, 1] == Cv)
    te_g = (test_groups[:, 0] == Rv) & (test_groups[:, 1] == Cv)

    if not np.any(te_g):
        continue

    group_baseline = end_insp_median_by_group.get((int(Rv), int(Cv)), np.nan)
    if pd.isna(group_baseline):
        group_baseline = global_exp_baseline
    else:
        group_baseline = float(find_nearest(float(group_baseline)))

    exp_rows = te_g & (~insp_mask_test_sorted)
    if np.any(exp_rows):
        test_pred_sorted[exp_rows] = np.float32(group_baseline)

    insp_rows = te_g & insp_mask_test_sorted
    if not np.any(insp_rows):
        continue
    if not np.any(tr_g):
        test_pred_sorted[insp_rows] = np.float32(group_baseline)
        continue

    n_tr = int(tr_g.sum())
    if n_tr < min_group_train_samples:
        test_pred_sorted[insp_rows] = np.float32(np.median(y_train[tr_g]))
        continue

    scaler_g = StandardScaler(with_mean=True, with_std=True)
    Xtr_g = scaler_g.fit_transform(X_train[tr_g])
    Xte_g = scaler_g.transform(X_test_sorted[insp_rows])

    model_g = Ridge(alpha=0.5, random_state=2021)
    model_g.fit(Xtr_g, y_train[tr_g])

    test_pred_sorted[insp_rows] = model_g.predict(Xte_g).astype(np.float32)

test_pred = np.empty_like(test_pred_sorted)
test_pred[test_feat["_orig_order"].values] = test_pred_sorted

pmin, pmax = float(sorted_pressures[0]), float(sorted_pressures[-1])
test_pred = np.clip(test_pred, pmin, pmax)
test_pred = snap_to_pressure_grid(test_pred)

submission = pd.DataFrame({"id": df_test["id"].values, "pressure": test_pred})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print("Pressure unique (sample):", np.unique(submission["pressure"].values)[:10])

   id  pressure
0   1  6.610815
1   2  6.681117
2   3  6.892024
3   4  7.524743
4   5  8.508973
Wrote submission.csv with shape: (603600, 2)
Pressure unique (sample): [0.91634152 0.98664367 1.05694581 1.1975501  1.40845654 1.47875869
 1.61936298 1.68966512 1.75996727 1.83026941]
